# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

In [3]:

xlsx_path = Path("Altered Meeting history.xlsx")  # <-- change if needed. One missing meeting name was under first entry, assuming it was corresponding to the other meetings adjacent

xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])  # Participants (first sheet)
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,Scholar,Alison,Ringel,PhD,NaN,NaN
1,"John Burke, BSc, PhD",Targeting Lipid Biology in Cancer,Participant,John,Burke,"BSc, PhD",University of Victoria,Professor of Biochemistry and Microbiology
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,Participant,Bart,Vanhaesebroeck,PhD,University College London,Professor of Cell Signaling
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,Participant,Christina,Mitchell,"MB BS, PhD",Monash Institute of Pharmaceutical Sciences,NaN
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,Participant,Neil,Vasan,"MD, PhD",Columbia University,Assistant Professor of Medicine
...,...,...,...,...,...,...,...,...
2375,"Jeffrey Ward, MD, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Scholar,Jeffrey,Ward,"MD, PhD",Washington University,NaN
2376,"David Barbie, MD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,David,Barbie,MD,Ambrosino Biotech Consulting LLC,Associate Professor of Medicine
2377,"David Shackelford, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,David,Shackelford,PhD,University of California Los Angeles,Professor of Medicine
2378,"Daniel Frigo, PhD",Uncovering new mechanisms of LKB1-mediated tum...,Participant,Daniel,Frigo,PhD,MD Anderson Cancer Center,Associate Professor


In [4]:

def _norm_token(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^\w\s\-]", "", s)     # drop punctuation (commas/periods/etc.)
    s = re.sub(r"\s+", " ", s)
    return s

def parse_name_parts(full_name: str):
    """
    Returns dict with: first, middle (optional), last.
    Handles: 'Last, First Middle', 'First Middle Last'
    Assumes dict input names are already cleaned of titles/credentials.
    """
    s = (full_name or "").strip()
    if not s:
        return {"first": "", "middle": "", "last": ""}

    if "," in s:
        last, rest = [x.strip() for x in s.split(",", 1)]
        parts = rest.split()
        first = parts[0] if parts else ""
        middle = parts[1] if len(parts) > 1 else ""
        return {"first": _norm_token(first), "middle": _norm_token(middle), "last": _norm_token(last)}

    parts = s.split()
    if len(parts) == 1:
        return {"first": "", "middle": "", "last": _norm_token(parts[0])}

    first = parts[0]
    last = parts[-1]
    middle = parts[1] if len(parts) > 2 else ""
    return {"first": _norm_token(first), "middle": _norm_token(middle), "last": _norm_token(last)}

def attendee_matches_pi(attendee_name: str, pi: dict) -> bool:
    """
    Strict-ish matching:
      - last name must match EXACTLY
      - first name must match EXACTLY (or PI first is initial matching attendee first initial)
      - if attendee has middle initial, allow PI middle to match initial (optional)
    This prevents substring hits like 'SHERRy' for 'Scherr'.
    """
    a = parse_name_parts(attendee_name)

    pi_first = _norm_token(pi.get("first_name") or "")
    pi_middle = _norm_token(pi.get("middle_name") or "")
    pi_last  = _norm_token(pi.get("last_name") or "")

    if not a["last"] or not a["first"] or not pi_last:
        return False

    if a["last"] != pi_last:
        return False

    # First name: exact OR initial match (to tolerate "Charles" vs "C")
    if pi_first:
        if a["first"] == pi_first:
            pass
        elif (len(pi_first) == 1 and pi_first == a["first"][:1]):
            pass
        else:
            return False
    else:
        return False

    # Optional middle handling: if attendee provided middle, require PI to be compatible
    if a["middle"]:
        a_mi = a["middle"][:1]
        if pi_middle:
            if pi_middle == a["middle"] or pi_middle[:1] == a_mi:
                return True
            return False
        # If PI has no middle, still accept (common in data)
        return True

    return True

def project_has_exact_attendee_pi(attendee_name: str, proj: dict) -> bool:
    """
    Returns True only if the project's PI roster includes the attendee as an exact match.
    """
    pis = proj.get("principal_investigators")
    if not isinstance(pis, list):
        return False
    for pi in pis:
        if isinstance(pi, dict) and attendee_matches_pi(attendee_name, pi):
            return True
    return False

In [5]:
_TITLE_RE = re.compile(r"^(dr\.?|prof\.?|mr\.?|ms\.?|mrs\.?)\s+", re.I)
# Strips trailing comma-separated credentials (extend list if you have others)
_CRED_RE = re.compile(r"(?:,?\s*(?:MD|M\.D\.|PhD|Ph\.D\.|DO|D\.O\.|MPH|MS|MSc|MBA|JD|DDS|DVM|RN))+$", re.I)

def normalize_meeting_title(x: str) -> str:
    if pd.isna(x): return None
    s = str(x).strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        s = s[1:-1].strip()
    return re.sub(r"\s+", " ", s)

def strip_titles_and_credentials(name: str) -> str:
    if pd.isna(name): return None
    s = str(name).strip()
    s = _TITLE_RE.sub("", s)      # leading "Dr.", "Prof.", etc.
    s = _CRED_RE.sub("", s)       # trailing ", MD, PhD" etc.
    return re.sub(r"\s+", " ", s).strip()

def split_chair_names(chairs_cell) -> list[str]:
    if pd.isna(chairs_cell): return []
    s = re.sub(r"\s+", " ", str(chairs_cell).strip())
    parts = [p.strip() for p in s.split(";") if p.strip()]
    chairs = []
    for p in parts:
        for item in re.split(r"\s+(?:and|&)\s+", p):
            item = item.strip()
            if not item: 
                continue
            item = re.sub(r"\s+of\s+.+$", "", item).strip()   # drop trailing institution
            item = strip_titles_and_credentials(item)         # drop titles/credentials
            if item:
                chairs.append(item)
    out, seen = [], set()
    for c in chairs:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

In [6]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

# Normalize meeting names
meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Strip titles/credentials from participant names
participants_df["Participant_clean"] = participants_df["Participant"].map(strip_titles_and_credentials)

# Map normalized meeting topic -> year (first non-null year per meeting)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year to participants
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

# Build dict: "Meeting Topic (Year)" -> sorted unique participant_clean names
def make_meeting_year_key(meeting_norm, year):
    if meeting_norm is None:
        return None
    if pd.isna(year):
        return f"{meeting_norm} (Year Unknown)"
    y = int(year) if float(year).is_integer() else year
    return f"{meeting_norm} ({y})"

tmp = participants_df.dropna(subset=["Meeting_norm", "Participant_clean"]).copy()
tmp["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(tmp["Meeting_norm"], tmp["Year"])]

meeting_attendees_dict = (
    tmp.groupby("MeetingYearKey")["Participant_clean"]
       .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
       .to_dict()
)

# optional: ensure chairs are included too (also title/credential stripped)
meetings_df["Chairs_clean"] = meetings_df["Meeting Chairs"].apply(split_chair_names)
for _, row in meetings_df.iterrows():
    topic = row.get("MeetingTopic_norm")
    year = row.get("Year")
    if not topic:
        continue
    key = make_meeting_year_key(topic, year)
    if key not in meeting_attendees_dict:
        meeting_attendees_dict[key] = []
    for chair in (row.get("Chairs_clean") or []):
        meeting_attendees_dict[key].append(chair)
    meeting_attendees_dict[key] = sorted(set(meeting_attendees_dict[key]))

# Preview
list(meeting_attendees_dict.items())

[('2005 Scholar Retreat (2005)',
  ['Alison Bertuch',
   'Anthony Letai',
   'Charles L. Sawyers',
   'Charles Sherr',
   'Christopher Bakkenist',
   'Craig Thompson',
   'David E. Fisher',
   'David Tuveson',
   'Edward Attiyeh',
   'Elsa Flores',
   'James Amatruda',
   'Jan Karlseder',
   'Kimryn Rathmell',
   'Masashi Narita',
   'Nabeel Bardeesy',
   'Norman Sharpless',
   'Scott Lowe']),
 ('2006 Scholar Retreat (2006)',
  ['Alison Bertuch',
   'Anthony Letai',
   'Benjamin B. Willia',
   'Christopher Bakkenist',
   'Ed Harlow',
   'Edward Attiyeh',
   'Elsa Flores',
   'Gerard Evan',
   'Ingo K. Mellinghoff',
   'James Amatruda',
   'Jan Karlseder',
   'Jean Wang',
   'John Kemshead',
   'Kimberly Kelly',
   'Kimryn Rathmell',
   'Masashi Narita',
   'Michael Safran',
   'Nabeel Bardeesy',
   'Norman Sharpless']),
 ('2007 Scholar Retreat (2007)',
  ['Anthony Letai',
   'Benjamin B. Willia',
   'Benjamin L. Ebert',
   'Carla F. Bender Kim',
   'Catriona Jamieson',
   'Edward Attiy

In [9]:

REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"

import os, json, hashlib, time
from copy import deepcopy

# Base raw-data directory
cwd = os.getcwd()
cwd_Raw_Data_outputs = os.path.join(cwd, "RawData")
os.makedirs(cwd_Raw_Data_outputs, exist_ok=True)
freeze_dir = os.path.join(cwd_Raw_Data_outputs, "NIH_Reporter_PI_FY")
os.makedirs(freeze_dir, exist_ok=True)


def _norm_name_for_match(s: str) -> str:
    s = str(s or "").strip().lower()
    s = re.sub(r"\([^)]*\)", "", s)            # remove parentheses
    s = re.sub(r"[^\w\s\-]", " ", s)           # strip punctuation
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _pi_full_name(pi: dict) -> str:
    # prefer full_name from API; else construct
    if not isinstance(pi, dict):
        return str(pi)
    full = (pi.get("full_name") or "").strip()
    if full:
        return re.sub(r"\s+", " ", full).strip()
    fn = (pi.get("first_name") or "").strip()
    mn = (pi.get("middle_name") or "").strip()
    ln = (pi.get("last_name") or "").strip()
    return re.sub(r"\s+", " ", f"{fn} {mn} {ln}").strip()

def non_attendee_pi_pairs(proj: dict, meeting_attendees: list[str]) -> list[tuple[str, int]]:
    """
    Return list of (PI full_name, profile_id) for PIs on this project
    that are NOT in the meeting attendee list.
    """
    pis = proj.get("principal_investigators")
    if not isinstance(pis, list):
        return []

    attendee_norm = {_norm_name_for_match(n) for n in meeting_attendees if isinstance(n, str) and n.strip()}
    out, seen = [], set()

    for pi in pis:
        if not isinstance(pi, dict):
            continue
        name = _pi_full_name(pi)
        pid = pi.get("profile_id")
        nname = _norm_name_for_match(name)

        # Skip PIs who are meeting attendees (including the queried attendee)
        if nname in attendee_norm:
            continue

        key = (nname, int(pid) if pid is not None else None)
        if key in seen:
            continue
        seen.add(key)
        out.append((name, int(pid) if pid is not None else None))

    return out
def _safe(s: str) -> str:
    s = str(s or "").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]", "", s)
    return s[:80] if s else "UNKNOWN"

def _freeze_path_for_pi_year(attendee_name: str, fiscal_year: int) -> str:
    # attendee_name expected already cleaned (no titles/credentials) from your dict
    # Works for "First Last" or "Last, First"
    name = attendee_name.strip()
    if "," in name:
        last, first = [x.strip() for x in name.split(",", 1)]
        first = first.split()[0] if first else ""
    else:
        toks = name.split()
        first = toks[0] if toks else ""
        last  = toks[-1] if toks else ""
    base = f"FY{int(fiscal_year)}__{_safe(last)}_{_safe(first)}"
    return os.path.join(freeze_dir, base + ".txt")

def reporter_search_all_pages_with_freeze(payload: dict, attendee_name: str, fiscal_year: int,
                                         sleep=0.25, limit=500, verbose=False):
    """
    Cache key is attendee_name + fiscal_year.
    Reads/writes ./RawData/NIH_Reporter_PI_FY/FY{year}__Last_First.txt
    """
    cache_file = _freeze_path_for_pi_year(attendee_name, fiscal_year)

    # ---- 1) read from freeze if exists ----
    if os.path.exists(cache_file):
        pages = []
        with open(cache_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("Date accessed:"):
                    continue
                pages.append(json.loads(line))
        total = pages[0].get("meta", {}).get("total", 0) if pages else 0
        if verbose:
            print(f"    [CACHE HIT] {os.path.basename(cache_file)} total={total}")
        return {"total": int(total or 0), "pages": pages, "from_cache": True, "cache_path": cache_file}

    # ---- 2) otherwise fetch + freeze ----
    params = deepcopy(payload)
    params.update({"offset": 0, "limit": limit})

    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json()
        pages.append(page)

        meta = page.get("meta", {})
        total = total if total is not None else meta.get("total", 0)
        off = meta.get("offset", params["offset"])
        cnt = meta.get("count", len(page.get("results", [])))

        if cnt == 0 or off + cnt >= total:
            break

        params["offset"] = off + cnt
        time.sleep(sleep)

    with open(cache_file, "w", encoding="utf-8") as f:
        for p in pages:
            f.write(json.dumps(p))
            f.write("\n")
        f.write(f"Date accessed: {datetime.now().isoformat()}\n")

    if verbose:
        print(f"    [CACHED] {os.path.basename(cache_file)} total={int(total or 0)}")

    return {"total": int(total or 0), "pages": pages, "from_cache": False, "cache_path": cache_file}
def meeting_year_from_key(meeting_key: str):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def extract_pi_names(pi_list):
    if not isinstance(pi_list, list): return ""
    out = []
    for pi in pi_list:
        if isinstance(pi, dict):
            fn = str(pi.get("first_name") or "").strip()
            ln = str(pi.get("last_name") or "").strip()
            full = str(pi.get("full_name") or "").strip()
            if ln and fn: out.append(f"{ln}, {fn}")
            elif full: out.append(full)
            elif ln: out.append(ln)
            elif fn: out.append(fn)
        else:
            out.append(str(pi))
    # de-dupe preserve order
    seen, dedup = set(), []
    for x in out:
        x = re.sub(r"\s+", " ", x).strip()
        if x and x not in seen:
            seen.add(x); dedup.append(x)
    return "; ".join(dedup)

def grants_by_attendee_year_bins_exact(
    meeting_attendees_dict: dict,
    first_n_meetings: int = 10,
    sleep: float = 0.25,
    verbose: bool = True
) -> pd.DataFrame:
    NIH_param_template = {"criteria": { "multi_pi_only": True }}#SEARCH CRITERIA HERE
    rows = []
    meeting_keys = list(meeting_attendees_dict.keys())[:first_n_meetings]

    print(f"Running NIH RePORTER attendee grant search (EXACT PI roster match) for first {len(meeting_keys)} meetings...")

    for mi, meeting_key in enumerate(meeting_keys, start=1):
        meeting_year = meeting_year_from_key(meeting_key)
        if meeting_year is None:
            if verbose: print(f"[{mi}/{len(meeting_keys)}] {meeting_key}: no year found, skipping")
            continue

        attendees = [n for n in meeting_attendees_dict.get(meeting_key, []) if isinstance(n, str) and n.strip()]
        if verbose:
            print(f"\n[{mi}/{len(meeting_keys)}] Meeting: {meeting_key} | Year={meeting_year} | Attendees={len(attendees)}")

        for y in range(meeting_year - 5, meeting_year + 10 + 1):
            if verbose: print(f"  FY={y} ...")

            for attendee in attendees:
                # Parse full first+last to use in query (narrower than last-only)
                a_parts = parse_name_parts(attendee)
                if not a_parts["last"] or not a_parts["first"]:
                    continue

                payload = deepcopy(NIH_param_template)
                c = payload["criteria"]

                # Still wildcarded by API, but we pass full first+last to reduce noise
                c["pi_names"] = [{"last_name": a_parts["last"], "first_name": a_parts["first"]}]
                c["fiscal_years"] = [int(y)]

                # Use working cache-based fetch
                res = reporter_search_all_pages_with_freeze(
                    payload,
                    attendee_name=attendee,
                    fiscal_year=int(y),
                    sleep=sleep,
                    limit=500,
                    verbose=False
                )
                if res["total"] <= 0:
                    continue

                kept = 0
                rejected = 0

                for page in res["pages"]:
                    for proj in page.get("results", []):
                        # exact PI roster check: keeps only true attendee matches
                        if not project_has_exact_attendee_pi(attendee, proj):
                            print("REJECTED")
                            print(attendee)
                            print(proj)
                            rejected += 1
                            continue

                        appl = proj.get("appl_id")
                        if appl is None:
                            rejected += 1
                            continue
                        
                        kept += 1
                        rows.append({
                            "MeetingYearKey": meeting_key,
                            "MeetingYear": meeting_year,
                            "BinFiscalYear": int(y),
                            "AttendeeQueried": attendee,
                            "ApplId": appl,
                            "AwardAmount": int(proj.get("award_amount") or 0),
                            "ProjectNum": proj.get("project_num"),
                            "ProjectTitle": proj.get("project_title"),
                            "PI_Names_On_Project": extract_pi_names(proj.get("principal_investigators")),
                            "NonAttendeePIs_NameProfilePairs": non_attendee_pi_pairs(proj, attendees),
                        })

                if verbose and (kept > 0 or rejected > 0):
                    print(f"    {attendee}: total={res['total']} kept_exact={kept} rejected_nonexact={rejected}")

                time.sleep(sleep)

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["MeetingYearKey","BinFiscalYear","AttendeeQueried","ApplId"])
    return df

In [ ]:
def all_pi_pairs(proj: dict) -> list[tuple[str, int]]:
    pis = proj.get("principal_investigators")
    if not isinstance(pis, list):
        return []
    out, seen = [], set()
    for pi in pis:
        if not isinstance(pi, dict):
            continue
        name = _pi_full_name(pi)
        pid = pi.get("profile_id")
        key = (_norm_name_for_match(name), int(pid) if pid is not None else None)
        if key in seen: 
            continue
        seen.add(key)
        out.append((name, int(pid) if pid is not None else None))
    return out
"AllPIs_NameProfilePairs": all_pi_pairs(proj),

In [8]:
attendee_grants_df = grants_by_attendee_year_bins_exact(
    meeting_attendees_dict=meeting_attendees_dict,
    first_n_meetings=150,
    sleep=0.25,
    verbose=True
)

attendee_grants_df.head(25)



Running NIH RePORTER attendee grant search (EXACT PI roster match) for first 147 meetings...

[1/147] Meeting: 2005 Scholar Retreat (2005) | Year=2005 | Attendees=17
  FY=2000 ...
  FY=2001 ...
  FY=2002 ...
  FY=2003 ...
  FY=2004 ...
  FY=2005 ...
  FY=2006 ...
  FY=2007 ...
  FY=2008 ...
  FY=2009 ...
    James Amatruda: total=1 kept_exact=1 rejected_nonexact=0
  FY=2010 ...
    James Amatruda: total=1 kept_exact=1 rejected_nonexact=0
REJECTED
Nabeel Bardeesy
REJECTED
Nabeel Bardeesy
    Nabeel Bardeesy: total=2 kept_exact=0 rejected_nonexact=2
  FY=2011 ...
REJECTED
Nabeel Bardeesy
REJECTED
Nabeel Bardeesy
    Nabeel Bardeesy: total=2 kept_exact=0 rejected_nonexact=2
  FY=2012 ...
REJECTED
Nabeel Bardeesy
REJECTED
Nabeel Bardeesy
    Nabeel Bardeesy: total=2 kept_exact=0 rejected_nonexact=2
    Norman Sharpless: total=1 kept_exact=1 rejected_nonexact=0
    Scott Lowe: total=1 kept_exact=1 rejected_nonexact=0
  FY=2013 ...
REJECTED
Nabeel Bardeesy
REJECTED
Nabeel Bardeesy
    Nabeel

KeyboardInterrupt: 

In [9]:

from pathlib import Path

def save_df_as_multiple_excels(df: pd.DataFrame, out_dir="excel_pages", base_name="attendee_grants",
                              rows_per_file=50000):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if df.empty:
        print("DataFrame is empty; nothing to save.")
        return []

    paths = []
    n = len(df)
    file_idx = 1
    for start in range(0, n, rows_per_file):
        end = min(start + rows_per_file, n)
        part = df.iloc[start:end].copy()
        out_path = out_dir / f"{base_name}_part{file_idx:03d}.xlsx"
        part.to_excel(out_path, index=False, engine="openpyxl")
        paths.append(str(out_path))
        print(f"Saved rows {start}-{end-1} -> {out_path}")
        file_idx += 1

    return paths

In [10]:
paths = save_df_as_multiple_excels(attendee_grants_df, out_dir="attendee_grants_excel_pages", rows_per_file=50000)


Saved rows 0-3621 -> attendee_grants_excel_pages\attendee_grants_part001.xlsx
